# Train + evaluate YOLO on the synthetic UR3 dataset (Colab)

Runtime > Change runtime type > **T4 GPU**. Dataset: zip in YOLO format (`images/`, `labels/`, `data.yaml`) produced from the Gazebo capture. See `docs/CV_DELIVERABLES.md`.

In [ ]:
!nvidia-smi
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/ur3_synth/dataset.zip'  # <- edit
!mkdir -p /content/ds && unzip -q -o $ZIP -d /content/ds
!ls /content/ds

In [ ]:
# data.yaml must contain train/val/test paths relative to `path` and the class names.
# Rewrite `path` so it points at the unzipped folder.
import yaml, pathlib
p = pathlib.Path('/content/ds/data.yaml')
cfg = yaml.safe_load(p.read_text())
cfg['path'] = '/content/ds'
p.write_text(yaml.safe_dump(cfg))
print(cfg)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/ds/data.yaml', epochs=50, imgsz=640, batch=16,
            seed=0, project='/content/drive/MyDrive/ur3_synth/runs', name='yolov8n_synth')

In [ ]:
# Held-out evaluation. mAP50 and mAP50-95 are the numbers for the report.
best = YOLO('/content/drive/MyDrive/ur3_synth/runs/yolov8n_synth/weights/best.pt')
m = best.val(data='/content/ds/data.yaml', split='test')
print('mAP50    :', m.box.map50)
print('mAP50-95 :', m.box.map)
for name, ap in zip(m.names.values(), m.box.maps):
    print(f'{name:12s} AP50-95 = {ap:.3f}')

## Using the weights in the ROS node
Copy `best.pt` to the repo and set the model path in `ur_perception` (it currently hardcodes `yolov8n.pt`, so that line needs to become a parameter).